In [1]:
!java-version

/bin/bash: line 1: java-version: command not found


In [2]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  openjdk-17-jre-headless
Suggested packages:
  openjdk-17-demo openjdk-17-source libnss-mdns fonts-dejavu-extra
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  openjdk-17-jdk-headless openjdk-17-jre-headless
0 upgraded, 2 newly installed, 0 to remove and 30 not upgraded.
Need to get 120 MB of archives.
After this operation, 272 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 openjdk-17-jre-headless amd64 17.0.20.1+1-1~24.04 [48.1 MB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 open

In [5]:
!java -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [4]:
!wget -q --show-progress \
  https://archive.apache.org/dist/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz


hadoop-3.5.0.tar.gz 100%[===================>] 554.35M  1.93MB/s    in 22m 4s  


In [6]:
!tar -xzf hadoop-3.5.0.tar.gz


In [7]:
import os
import subprocess

os.environ["HADOOP_HOME"] = "/content/hadoop-3.5.0"
os.environ["JAVA_HOME"] = os.path.realpath(
    subprocess.check_output(
        ["bash", "-lc", "dirname $(dirname $(readlink -f $(which java)))"],
        text=True
    ).strip()
)
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin:" + os.environ["PATH"]

with open(os.environ["HADOOP_HOME"] + "/etc/hadoop/hadoop-env.sh", "a") as f:
    f.write("\nexport JAVA_HOME=" + os.environ["JAVA_HOME"] + "\n")

!hadoop version


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hadoop 3.5.0
Source code repository https://github.com/apache/hadoop.git -r dbcc7cd797100e6b32cd84f85b53a5193a5f9af0
Compiled by cnauroth on 2026-03-24T16:56Z
Compiled on platform linux-x86_64
Compiled with protoc 3.25.5
From source with checksum 64582ed0a62a169c39a19c734a8f33b5
This command was run using /content/hadoop-3.5.0/share/hadoop/common/hadoop-common-3.5.0.jar


In [13]:
!hadoop version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hadoop 3.5.0
Source code repository https://github.com/apache/hadoop.git -r dbcc7cd797100e6b32cd84f85b53a5193a5f9af0
Compiled by cnauroth on 2026-03-24T16:56Z
Compiled on platform linux-x86_64
Compiled with protoc 3.25.5
From source with checksum 64582ed0a62a169c39a19c734a8f33b5
This command was run using /content/hadoop-3.5.0/share/hadoop/common/hadoop-common-3.5.0.jar


In [8]:
!unzip hadoop_lab_data_and_code.zip

Archive:  hadoop_lab_data_and_code.zip
  inflating: README.txt              
  inflating: orders_2026_09.csv      
  inflating: reducer_revenue.py      
  inflating: mapper_revenue.py       


In [9]:
!cat orders_2026_09.csv | python3 mapper_revenue.py | sort | python3 reducer_revenue.py

Beauty	111032003
Book	39306022
Electronics	3703769311
Fashion	218597763
Grocery	80983147
Home	410315065
Sport	166194354
Toy	83881779


In [10]:
import glob
import shutil
import subprocess

# Xóa thư mục kết quả cũ nếu có để tránh lỗi "Output already exists"
shutil.rmtree("/content/output_revenue", ignore_errors=True)

jar = glob.glob(
    "/content/hadoop-3.5.0/share/hadoop/tools/lib/hadoop-streaming*.jar")[0]

cmd = [
    "hadoop", "jar", jar,
    "-D", "mapreduce.framework.name=local",
    "-D", "fs.defaultFS=file:///",
    "-files", "/content/mapper_revenue.py,/content/reducer_revenue.py",
    "-mapper", "python3 mapper_revenue.py",
    "-reducer", "python3 reducer_revenue.py",
    "-input", "file:///content/orders_2026_09.csv",
    "-output", "file:///content/output_revenue"
]

subprocess.run(cmd, check=True)

CompletedProcess(args=['hadoop', 'jar', '/content/hadoop-3.5.0/share/hadoop/tools/lib/hadoop-streaming-3.5.0.jar', '-D', 'mapreduce.framework.name=local', '-D', 'fs.defaultFS=file:///', '-files', '/content/mapper_revenue.py,/content/reducer_revenue.py', '-mapper', 'python3 mapper_revenue.py', '-reducer', 'python3 reducer_revenue.py', '-input', 'file:///content/orders_2026_09.csv', '-output', 'file:///content/output_revenue'], returncode=0)

In [11]:
!cat /content/output_revenue/part-*
!sort -k2,2nr /content/output_revenue/part-* | head -3

Beauty	111032003
Book	39306022
Electronics	3703769311
Fashion	218597763
Grocery	80983147
Home	410315065
Sport	166194354
Toy	83881779
Electronics	3703769311
Home	410315065
Fashion	218597763


In [14]:
from google.colab import files
files.download('/content/output_revenue/part-00000')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>